# Module 4 — Adaptive Learning Verification Agent

In [ ]:
!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai
import json

GEMINI_API_KEY = ""
MODEL_NAME     = "gemini-2.0-flash"

genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
with open('assessment_fractions.json', 'r', encoding='utf-8') as f:
    previous_assessment = json.load(f)

with open('diagnostic_report_fractions.json', 'r', encoding='utf-8') as f:
    gap_json = json.load(f)

In [ ]:
def build_retest_prompt(gap_json, previous_assessment, num_questions=7):
    weak_skills = '\n\n'.join(
        f"{i+1}. {g['micro_skill']}\n"
        f"   Misconception: {g['misconception']}\n"
        f"   Severity: {g['severity']} | Priority: {g['recommended_priority']}\n"
        f"   Root cause: {g['root_cause']}"
        for i, g in enumerate(gap_json['identified_gaps'])
    )

    previous_questions = '\n'.join(
        f"- Q{q['id']}: \"{q['question']}\" [{q['micro_skill']}]"
        for q in previous_assessment['questions']
    )

    easy = round(num_questions * 0.3)
    medium = round(num_questions * 0.4)
    hard = num_questions - easy - medium

    return f"""You are an Adaptive Learning Verification Agent specializing in targeted reassessment for K-12 students.

Your task is to generate a focused diagnostic re-assessment that ONLY targets the student's identified weak micro-skills.

Subject: {previous_assessment['subject']}
Topic: {previous_assessment['topic']}
Grade: {previous_assessment['grade']}
Mastery Level: {gap_json['mastery_level']}
Overall Score: {gap_json['overall_score']}/100

Weak Micro-Skills to Target:
{weak_skills}

RULES:
1. ONLY generate questions targeting the weak micro-skills listed above.
2. NEVER repeat or paraphrase any of these previous questions:
{previous_questions}

3. Generate exactly {num_questions} questions: {easy} Easy, {medium} Medium, {hard} Hard.
4. If mastery_level is "Beginner" or "Needs Immediate Intervention" — start with Easy questions and increase gradually.
5. If mastery_level is "Intermediate" — skip trivial Easy, begin from slightly harder Easy.
6. Each question must target a different distractor/misconception than the original assessment used.
7. Every question must test the SAME micro-skill but from a different angle.

Return ONLY valid JSON. No markdown. No explanations outside JSON.

{{
  "subject": "{previous_assessment['subject']}",
  "topic": "{previous_assessment['topic']}",
  "grade": "{previous_assessment['grade']}",
  "curriculum": "{previous_assessment.get('curriculum', '')}",
  "language": "{previous_assessment.get('language', 'English')}",
  "assessment_metadata": {{
    "difficulty_distribution": {{"Easy": "{easy}/{num_questions}", "Medium": "{medium}/{num_questions}", "Hard": "{hard}/{num_questions}"}},
    "estimated_completion_minutes": 0
  }},
  "questions": [
    {{
      "id": 1,
      "micro_skill": "",
      "concept_category": "",
      "difficulty": "Easy",
      "blooms_level": "Understand",
      "question": "",
      "options": {{ "A": "", "B": "", "C": "", "D": "" }},
      "correct_option": "A",
      "misconception_tested": "",
      "explanation": "",
      "distractor_analysis": {{ "A": "", "B": "", "C": "", "D": "" }},
      "estimated_time_seconds": 45,
      "grade_alignment_confidence": 0.97
    }}
  ]
}}"""

In [ ]:
def generate_retest(gap_json, previous_assessment, num_questions=7):
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        generation_config={
            'response_mime_type': 'application/json',
            'temperature': 0.2,
            'max_output_tokens': 6144,
        }
    )
    prompt = build_retest_prompt(gap_json, previous_assessment, num_questions)
    response = model.generate_content(prompt)
    return json.loads(response.text)

In [ ]:
retest_assessment = generate_retest(gap_json, previous_assessment, 7)
print(len(retest_assessment['questions']))

In [ ]:
def weighted_score(questions, responses):
    weights = {"Easy": 1, "Medium": 2, "Hard": 3}
    q_map = {q['id']: q for q in questions}
    earned = 0
    max_pts = 0
    for r in responses:
        q = q_map.get(r['question_id'])
        if not q: continue
        w = weights.get(q['difficulty'], 1)
        max_pts += w
        if r['selected_option'] == q['correct_option']: earned += w
    return round((earned / max_pts) * 100) if max_pts > 0 else 0

def build_eval_prompt(initial_assessment, initial_responses, retest_assessment, retest_responses, gap_json):
    init_score = weighted_score(initial_assessment['questions'], initial_responses)
    ret_score = weighted_score(retest_assessment['questions'], retest_responses)
    
    def format_responses(assessment, responses, label):
        q_map = {q['id']: q for q in assessment['questions']}
        res_lines = []
        for r in responses:
            q = q_map.get(r['question_id'])
            if not q: continue
            correct = r['selected_option'] == q['correct_option']
            status = "CORRECT" if correct else f"INCORRECT (chose {r['selected_option']}, correct {q['correct_option']})"
            res_lines.append(f"{label} Q{q['id']} [{q['difficulty']}] {q['micro_skill']}: {status}")
        return '\n'.join(res_lines)

    init_block = format_responses(initial_assessment, initial_responses, "Initial")
    ret_block = format_responses(retest_assessment, retest_responses, "Retest")

    orig_gaps = '\n'.join(f"- {g['micro_skill']} [{g['severity']}]" for g in gap_json['identified_gaps'])

    return f"""You are an Adaptive Learning Verification Agent specializing in measuring student learning progress.

Subject: {initial_assessment['subject']}
Topic: {initial_assessment['topic']}
Grade: {initial_assessment['grade']}
Initial Mastery Level: {gap_json['mastery_level']}

Initial Assessment — Weighted Score: {init_score}/100
Retest Assessment  — Weighted Score: {ret_score}/100

Initial Responses:
{init_block}

Retest Responses:
{ret_block}

Original Gaps Identified:
{orig_gaps}

ANALYSIS RULES:
1. Calculate improvement_score as: ((retest_weighted_score - initial_weighted_score) / (100 - initial_weighted_score)) * 100, clamped to 0–100.
   If initial score was 100, improvement_score = 100.
2. A gap is "resolved" if the student answered all retest questions for that micro-skill correctly.
   A gap "remains" if they still got at least one question wrong for that micro-skill.
3. ready_for_next_topic = true if retest_weighted_score >= 70 AND remaining_gaps has no "High" severity items.
4. remediation_needed = true if retest_weighted_score < 50 OR remaining_gaps length >= 2 with High/Critical priority.
5. recommended_next_topics should list 2–4 logically sequential topics that build on the current one.
   If remediation_needed is true, recommend revisiting sub-topics of the current one instead.
6. assessment_summary: 2–3 sentence narrative summary of the student's journey from initial to retest.

Return ONLY valid JSON. No markdown. No explanations outside JSON.

{{
  "improvement_score": 0,
  "initial_weighted_score": {init_score},
  "retest_weighted_score": {ret_score},
  "remaining_gaps": [""],
  "resolved_gaps": [""],
  "ready_for_next_topic": true,
  "recommended_next_topics": [""],
  "remediation_needed": false,
  "assessment_summary": ""
}}"""

In [ ]:
def evaluate_progress(initial_assessment, initial_responses, retest_assessment, retest_responses, gap_json):
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        generation_config={
            'response_mime_type': 'application/json',
            'temperature': 0.2,
            'max_output_tokens': 2048,
        }
    )
    prompt = build_eval_prompt(initial_assessment, initial_responses, retest_assessment, retest_responses, gap_json)
    response = model.generate_content(prompt)
    return json.loads(response.text)

In [ ]:
initial_responses = [
    {"question_id": 1, "selected_option": "A"},
    {"question_id": 2, "selected_option": "C"},
    {"question_id": 3, "selected_option": "B"},
    {"question_id": 4, "selected_option": "A"},
    {"question_id": 5, "selected_option": "D"},
    {"question_id": 6, "selected_option": "B"},
    {"question_id": 7, "selected_option": "A"},
    {"question_id": 8, "selected_option": "C"},
    {"question_id": 9, "selected_option": "D"},
    {"question_id": 10, "selected_option": "B"}
]

retest_responses = []
for q in retest_assessment['questions']:
    retest_responses.append({"question_id": q['id'], "selected_option": q['correct_option']})
retest_responses[-1]['selected_option'] = "D"

In [ ]:
evaluation = evaluate_progress(previous_assessment, initial_responses, retest_assessment, retest_responses, gap_json)
print(evaluation['improvement_score'])
print(evaluation['ready_for_next_topic'])
print(evaluation['assessment_summary'])

In [ ]:
with open('retest_assessment.json', 'w', encoding='utf-8') as f:
    json.dump(retest_assessment, f, indent=2, ensure_ascii=False)

with open('progress_evaluation.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation, f, indent=2, ensure_ascii=False)